# Lesson 4.1 — Explicit Criteria to Reduce False Positives

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain why explicit categorical criteria beat vague instructions like 'be conservative'
- Write review criteria that define what to report vs skip
- Use concrete code examples per severity level for consistent classification
- Temporarily disable high-false-positive categories to restore developer trust

**You will produce:** A review-criteria spec with report/skip rules and example-anchored severity levels.

## Phase 4 Overview

Phase 4 covers **Domain 4 - Prompt Engineering & Structured Output (20%)**. You will write explicit criteria, apply few-shot prompting, enforce schema-compliant output via tool use, build validation/retry loops, design batch strategies, and architect multi-pass reviews.

By the end of this phase, you will have built:

- Explicit review criteria that cut false positives (Lesson 4.1)
- Few-shot examples for ambiguous cases (Lesson 4.2)
- tool_use + JSON schema structured output with tool_choice (Lesson 4.3)
- Validation/retry/feedback loops (Lesson 4.4)
- Message Batches API strategy with custom_id (Lesson 4.5)
- Multi-instance / multi-pass review architecture (Lesson 4.6)

### Prerequisites

- Phases 1-3 complete
- Python 3 kernel with the `anthropic` SDK; optional `ANTHROPIC_API_KEY`
- Familiarity with JSON Schema and Pydantic-style validation

## Concepts

**Explicit criteria over vague instructions.** "Flag comments only when claimed behavior contradicts actual code behavior" works; "check that comments are accurate" does not. General instructions like "be conservative" or "only report high-confidence findings" do **not** improve precision - specific **categorical criteria** do.

High false-positive categories undermine trust in the accurate ones. Techniques:
- **Define what to report** (bugs, security) **vs skip** (minor style, local patterns) rather than relying on confidence-based filtering.
- **Temporarily disable** a high-false-positive category to restore developer trust while you improve its prompt.
- **Anchor each severity** with concrete code examples for consistent classification.

Sample Question 3's lesson generalizes: unclear decision boundaries, not infrastructure, are usually the root cause - fix the criteria first.

> **Reference:** [Anthropic - prompt engineering (be clear & direct)](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Report/skip criteria instead of 'be conservative'

We encode categorical rules the model can apply deterministically.

In [ ]:
REVIEW_CRITERIA = {
    "report": ["logic bugs", "security vulnerabilities",
               "comment contradicts code behavior"],
    "skip": ["minor style", "local naming patterns", "subjective preferences"],
}
def criteria_prompt(c):
    rep = "; ".join(c["report"]); sk = "; ".join(c["skip"])
    return f"REPORT only: {rep}.\nSKIP: {sk}.\nDo NOT use 'be conservative' heuristics."

print(criteria_prompt(REVIEW_CRITERIA))

## Code-Along: Step 2 — Example-anchored severity levels

Each severity carries a concrete code example so classification is consistent.

In [ ]:
SEVERITY = {
    "critical": "SQL built via string concat with user input (injection)",
    "high": "Unhandled None deref on a hot path",
    "medium": "Missing input validation with low blast radius",
    "low": "Inconsistent log message formatting",
}
for sev, example in SEVERITY.items():
    print(f"{sev}: e.g. {example}")

## Code-Along: Step 3 — Disable a noisy category

Turn off a high-false-positive category while you refine it, keeping trust elsewhere.

In [ ]:
def active_categories(fp_rates, threshold=0.3):
    return {cat: ('on' if rate < threshold else 'disabled (refine prompt)')
            for cat, rate in fp_rates.items()}

print(active_categories({"security": 0.05, "style": 0.6, "perf": 0.2}))

## Exercise

Write `should_report(finding)` using REVIEW_CRITERIA that returns True only for reportable categories. Add a 'comment-accuracy' category with the explicit rule 'claimed behavior contradicts actual code' and assert a cosmetic comment mismatch is NOT reported while a behavioral contradiction IS.

In [ ]:
# TODO: implement should_report(finding) against REVIEW_CRITERIA
# TODO: assert should_report({'category': 'minor style'}) is False
# TODO: assert should_report({'category': 'comment contradicts code behavior'}) is True

## Key Takeaways

- Explicit categorical criteria reduce false positives; 'be conservative' does not.
- Define what to report vs skip instead of confidence-based filtering.
- Anchor each severity level with concrete code examples for consistency.
- Temporarily disable noisy categories to protect trust while you refine them.

---

**Next:** [Lesson 4.2 — Few-Shot Prompting for Consistency](./lesson-4.2-few-shot-prompting.ipynb)